### Retrieval-Augmented Generation (RAG) Chatbot

A RAG chatbot connects a large language model (LLM) to data sources like PDFs and websites. Doing so allows the chatbot to give context-aware and accurate answers to frequently-asked questions.

### Prepare Data

- Load the Data Digest PDF, extract text content, and clean text content.
- Chunking: Split documents into smaller, semantically-meaningful chunks (~500-1000 tokens).

In [ ]:
# Import libraries
import PyPDF2
import re

# Load and clean data_digest_2025.pdf
cleaned_text = []
with open('data_digest_2025.pdf', 'rb') as file:
    reader = PyPDF2.PdfReader(file)
    total_pages = len(reader.pages)
    
    # Process pages, skipping the first 5 (title, TOC, etc.)
    for page_num in range(5, total_pages):
        page = reader.pages[page_num]
        text = page.extract_text()
        
        # Skip table of contents pages (contain lots of dots)
        if "......" in text or ". . . . . ." in text or "Table of Contents" in text:
            continue
        
        # Remove headers, footers, and page numbers
        lines = text.split('\n')
        cleaned_lines = []
        
        for line in lines:
            # Skip empty lines
            if not line.strip():
                continue
            
            # Skip lines that are just page numbers
            if line.strip().isdigit():
                continue
            
            # Skip common header/footer patterns
            if re.match(r'^(Data Digest|University|Page \d+)', line.strip()):
                continue
            
            cleaned_lines.append(line.strip())
        
        # Only add pages with substantial content
        cleaned_page_text = '\n'.join(cleaned_lines)
        if len(cleaned_page_text) > 50:  # Skip nearly-empty pages
            cleaned_text.append(cleaned_page_text)

# Display summary
print(f"Total pages in PDF: {total_pages}")
print(f"Pages processed and cleaned: {len(cleaned_text)}")
print(f"\nSample from first cleaned page:")
print(cleaned_text[0][:300] if cleaned_text else "No text extracted")

Total pages in PDF: 147
Pages skipped (first 5): 5
Pages processed and cleaned: 135

Sample from first cleaned page:
DATA, ACADEMIC PLANNING, AND INSTITUTIONAL RESEARCH, OFFICE OF THE PROVOSTUniversity of Wisconsin–Madison 2024 – 2025  DATA DIGEST     |    Students
1010,00020,00030,00040,00050,00060,000
1888 1896 1904 1912 1920 1928 1936 1944 1952 1960 1968 1976 1984 1992 2000 2008 2016 2024Students Enrolled
YearT


In [24]:
# Check results - display first few cleaned pages
print(f"Total cleaned pages: {len(cleaned_text)}\n")

for i in range(min(3, len(cleaned_text))):
    print(f"{'='*60}")
    print(f"CLEANED PAGE {i + 1}")
    print(f"{'='*60}")
    print(cleaned_text[i][:500])  # Show first 500 characters
    print(f"\n... [truncated, full page length: {len(cleaned_text[i])} characters]\n")

Total cleaned pages: 135

CLEANED PAGE 1
DATA, ACADEMIC PLANNING, AND INSTITUTIONAL RESEARCH, OFFICE OF THE PROVOSTUniversity of Wisconsin–Madison 2024 – 2025  DATA DIGEST     |    Students
1010,00020,00030,00040,00050,00060,000
1888 1896 1904 1912 1920 1928 1936 1944 1952 1960 1968 1976 1984 1992 2000 2008 2016 2024Students Enrolled
YearTotal Enrollments from 1888 through 2024
FemaleTotal
Male

... [truncated, full page length: 356 characters]

CLEANED PAGE 2
DATA, ACADEMIC PLANNING, AND INSTITUTIONAL RESEARCH, OFFICE OF THE PROVOSTUniversity of Wisconsin–Madison 2024 – 2025  DATA DIGEST     |    Students     2
Student Level 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024
Undergraduate 29,580 29,536 29,931 30,360 31,185 31,650 33,506 35,184 35,665 36,902
Freshmen 4,685 4,860 5,022 4,909 5,359 5,353 6,489 6,188 5,553 5,267
Sophomores 6,540 6,272 6,471 6,778 6,800 6,982 7,230 8,609 8,486 8,772
Juniors 7,799 7,746 7,642 7,941 7,998 8,117 8,538 8,815 9

... [truncated, full page length:

### Convert and Store Embeddings

- Convert text chunks into vector embeddings with a pre-trained model (HuggingFace sentence-transformers).
- Store embeddings in a vector database (ChromaDB) to index chunks with embeddings and metadata.

### Retrieval

- Convert user prompts into embeddings using the same, pre-trained model.
- Execute similarity search to find the most relevant text chunks.

### Generation

- Pass the retrieved chunks as **context** to an open-source LLM (Llamma).
- LLM prompts = user prompt + retrieved context + sys. instructions
- Generate an answer grounded in the source documents.
- Optionally include references to source documents.

### Model Evaluation

- Test the chatbot with sample FAQs.
- Are the answers accurate and helpful?